# 02 · Train the keyword model

Trains a model that hears one second of audio and answers with one of

`_silence_` · `_unknown_` · `crystal` · `on` · `off` · `bright`

then converts it to an int8 TensorFlow Lite model and a C array for the Arduino.
Run `01_prepare_data.ipynb` first.

Two architectures are trained and compared:

- **tiny_conv**: the model of the `micro_speech` example. One convolution, one dense layer.
- **ds_cnn**: a depthwise-separable CNN, the keyword spotting reference of the MLPerf Tiny benchmark.
  More layers and roughly ten times the computation.

On a laptop CPU the whole notebook takes about 20 to 30 minutes, most of it for `ds_cnn`.

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's C++ info messages

In [ ]:
import json

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

from fairy_kws import config, dataset, export, models

LABELS = config.LABELS
ARCHITECTURES = ["tiny_conv", "ds_cnn"]
BATCH_SIZE = 100
STEPS_PER_EPOCH = 400
MAX_EPOCHS = 40
SEED = 0

keras.utils.set_random_seed(SEED)
print("labels:", LABELS)

## 1. Data

**Training data** is an endless stream. Each example is drawn from a random class, so a word
with few clips shows up as often as a word with many. Every clip is shifted by up to 100 ms and mixed
with background noise at a random volume, so the model never sees exactly the same input twice.
The `_unknown_` class gets double weight because, in a living room, almost everything the lamp
hears is not a command. Within each class the two datasets are drawn from equally, so half of the
unknown examples are MSWC words, many of them picked because they sound like our keywords
(*Christmas*, *light*, *upon*, *office*).

**Validation and test data** are fixed: all keyword clips of the split, the same number of unknown
clips, and some silence (background noise only). The speakers in them do not appear in the
training data. Each clip is mixed with background noise in the same way as in training, but with a
fixed random seed. The MSWC clips (`crystal`, `bright`, part of `off`) are padded with digital
silence, which no microphone produces, so testing them without noise would flatter the model. A
clean copy of the test set is kept for comparison.

In [ ]:
clips = dataset.build_clip_table()
train_counts = clips[clips["split"] == "train"].groupby("label").size()
print("training clips per label:")
print(train_counts.to_string())

train_ds = dataset.training_dataset(
    clips, LABELS, class_weights={config.UNKNOWN_LABEL: 2.0}, batch_size=BATCH_SIZE, seed=SEED
)
x_val, y_val, val_table = dataset.evaluation_set(clips, "val", LABELS)
x_test, y_test, test_table = dataset.evaluation_set(clips, "test", LABELS)
x_test_clean, _, _ = dataset.evaluation_set(clips, "test", LABELS, background=False)
print("\nvalidation:", x_val.shape, " test:", x_test.shape)
print("test examples per label:", dict(zip(LABELS, np.bincount(y_test).tolist(), strict=True)))

In [ ]:
batch_x, batch_y = next(iter(train_ds))
fig, axes = plt.subplots(2, 6, figsize=(13, 4), constrained_layout=True)
for ax, features, label in zip(axes.flat, batch_x, batch_y, strict=False):
    ax.imshow(features[..., 0].numpy().T, origin="lower", aspect="auto")
    ax.set_title(LABELS[int(label)], fontsize=10)
    ax.axis("off")
fig.suptitle("One training batch (first 12 examples, after augmentation)")
plt.show()

## 2. Train

Both models train the same way: Adam, learning rate halved when validation loss stops improving,
and early stopping that keeps the weights of the best validation epoch. One "epoch" here is simply
400 batches of the endless stream.

In [ ]:
def train(name):
    model = models.ARCHITECTURES[name](len(LABELS))
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    history = model.fit(
        train_ds,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=MAX_EPOCHS,
        validation_data=(x_val, y_val),
        callbacks=[
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5
            ),
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=8, restore_best_weights=True
            ),
        ],
        shuffle=False,  # the stream is already shuffled
        verbose=2,
    )
    return model, history.history


trained, histories = {}, {}
for name in ARCHITECTURES:
    print(f"\n===== {name} =====")
    trained[name], histories[name] = train(name)

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for name, history in histories.items():
    ax_loss.plot(history["val_loss"], label=f"{name} (validation)")
    ax_loss.plot(history["loss"], linestyle=":", label=f"{name} (training)")
    ax_acc.plot(history["val_accuracy"], label=f"{name} (validation)")
    ax_acc.plot(history["accuracy"], linestyle=":", label=f"{name} (training)")
ax_loss.set(title="Loss", xlabel="epoch")
ax_acc.set(title="Accuracy", xlabel="epoch")
ax_acc.legend()
plt.show()

Training clips are time-shifted and the training stream has a different class mix (balanced)
than the validation set (half keywords, about 40% unknown), so the two accuracy curves are not
directly comparable.

## 3. Convert to int8 TensorFlow Lite

The microcontroller runs integer arithmetic only. The converter needs example inputs to choose the
integer ranges; we give it 500 training examples. From here on, every number is measured on the
**converted int8 model**, because that is the model that ships.

In [ ]:
representative = np.concatenate([x.numpy() for x, _ in train_ds.take(5)])

tflite_models, scores = {}, {}
rows = []
for name, model in trained.items():
    tflite_models[name] = export.to_int8_tflite(model, representative)
    scores[name] = export.tflite_predict(tflite_models[name], x_test)
    float_accuracy = (model.predict(x_test, verbose=0).argmax(axis=1) == y_test).mean()
    clean_scores = export.tflite_predict(tflite_models[name], x_test_clean)
    rows.append(
        {
            "model": name,
            "parameters": model.count_params(),
            "size (kB)": round(len(tflite_models[name]) / 1024, 1),
            "test accuracy, float": round(float(float_accuracy), 4),
            "test accuracy, int8": round(float((scores[name].argmax(axis=1) == y_test).mean()), 4),
            "int8, clean clips": round(float((clean_scores.argmax(axis=1) == y_test).mean()), 4),
            "operations": ", ".join(export.tflite_ops(tflite_models[name])),
        }
    )
summary = pd.DataFrame(rows).set_index("model")
summary

"Test accuracy" is on the test set with background noise; "clean clips" is the same set without
added noise, for comparison. The **operations** column matters for the firmware: `micro_speech` only registers the operations its
own model uses, so the sketch's op resolver has to list exactly these.

## 4. Results on the test set

Rows are what was said, columns what the model answered. *Recall* is the share of a word's clips
the model got right; *precision* is how often the model was right when it gave that answer.

In [ ]:
fig, axes = plt.subplots(1, len(scores), figsize=(6.5 * len(scores), 5.5), constrained_layout=True)
for ax, (name, model_scores) in zip(np.atleast_1d(axes), scores.items(), strict=True):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        model_scores.argmax(axis=1),
        display_labels=LABELS,
        normalize="true",
        values_format=".2f",
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"{name} (int8), share of each true label")
    ax.tick_params(axis="x", rotation=45)
plt.show()

for name, model_scores in scores.items():
    print(f"===== {name} (int8) =====")
    print(
        classification_report(
            y_test, model_scores.argmax(axis=1), target_names=LABELS, digits=3, zero_division=0
        )
    )

`off` comes from both datasets, so it shows whether the model does equally well on the two kinds of
recording (isolated words in Speech Commands, words cut from sentences in MSWC).

In [ ]:
by_source = test_table.assign(
    **{name: model_scores.argmax(axis=1) == y_test for name, model_scores in scores.items()}
)
by_source.groupby(["label", "source"])[list(scores)].agg(["mean", "size"]).round(3)

## 5. Detection threshold

On the board a keyword only counts when its score is above a threshold. Raising the threshold
trades missed commands for fewer false triggers. For each threshold:

- **keyword recall**: share of keyword clips where the right keyword scored above the threshold;
- **false triggers**: share of unknown and silence clips where *any* keyword scored above it.

This is measured on isolated one-second clips. The real figure of merit, false triggers per hour of
continuous sound, comes later with streaming tests.

In [ ]:
keyword_ids = [LABELS.index(k) for k in config.KEYWORDS]
is_keyword = np.isin(y_test, keyword_ids)

rows = []
for name, model_scores in scores.items():
    true_score = model_scores[np.arange(len(y_test)), y_test]
    best_keyword_score = model_scores[:, keyword_ids].max(axis=1)
    for threshold in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
        row = {"model": name, "threshold": threshold}
        row["keyword recall"] = (true_score[is_keyword] >= threshold).mean()
        for keyword in config.KEYWORDS:
            mask = y_test == LABELS.index(keyword)
            row[f"recall {keyword}"] = (true_score[mask] >= threshold).mean()
        row["false triggers"] = (best_keyword_score[~is_keyword] >= threshold).mean()
        rows.append(row)
thresholds = pd.DataFrame(rows).set_index(["model", "threshold"]).round(3)
thresholds

## 6. Export for the firmware

For each model, `models/<wake word>/<name>/` receives (one folder per wake word, so results for
different wake words can be compared):

- `model.keras`: the trained Keras model,
- `model_int8.tflite`: the converted model,
- `model.cc`: the same bytes as a C array (`g_model`), ready to replace the model file of `micro_speech`,
- `labels.txt`: label order, which must match `kCategoryLabels` in the sketch,
- `metrics.json`: the numbers from this notebook.

In [ ]:
for name, model in trained.items():
    out_dir = config.MODELS_DIR / config.WAKE_WORD / name
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save(out_dir / "model.keras")
    (out_dir / "model_int8.tflite").write_bytes(tflite_models[name])
    export.write_c_source(tflite_models[name], out_dir / "model.cc")
    (out_dir / "labels.txt").write_text("\n".join(LABELS) + "\n")
    metrics = {
        **summary.loc[name].to_dict(),
        "labels": LABELS,
        "epochs_trained": len(histories[name]["loss"]),
        # rows: true label, columns: predicted label, in the order of "labels"
        "confusion_matrix": confusion_matrix(y_test, scores[name].argmax(axis=1)).tolist(),
        "thresholds": thresholds.loc[name].reset_index().to_dict(orient="records"),
    }
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print(f"{name}: wrote {sorted(p.name for p in out_dir.iterdir())}")

labels_c = ", ".join(f'"{label}"' for label in LABELS)
print("\nFor micro_features_micro_model_settings:")
print(f"constexpr int kCategoryCount = {len(LABELS)};")
print(f"const char* kCategoryLabels[kCategoryCount] = {{{labels_c}}};")